# Perf-AI Copilot: An Enterprise Agent System for Automated Performance Analysis

This notebook contains the complete implementation of the Perf-AI Copilot, a multi-agent system designed to automate performance test analysis.

**Project Goal:** To reduce the time and expertise required to analyze performance test results by automating log parsing, metric calculation, root cause analysis, and report generation.

**Track:** Enterprise Agents


<h1 align="center">Perf-AI Copilot</h1>
<p align="center">
  An AI-powered multi-agent system that automates performance test analysis, reducing analysis time from hours to minutes.
</p>
<p align="center">
  <em>Submission for the Kaggle x Google Agents Intensive Capstone Project (Enterprise Agents Track).</em>
</p>

---

## 🎯 The Problem: The Performance Engineering Bottleneck

In any enterprise software development lifecycle, performance engineering is a critical but notoriously time-consuming process. After every performance test, engineering teams spend **6-10 hours** manually:

- **Parsing Complex Data:** Sifting through thousands of lines of raw logs from tools like JMeter.
- **Analyzing Metrics:** Inspecting dozens of graphs for CPU utilization, memory, response times, and error rates.
- **Identifying Bottlenecks:** Correlating metric spikes with application behavior to pinpoint the root cause of slowdowns.
- **Generating Reports:** Compiling findings, charts, and recommendations into detailed reports for stakeholders.

This manual cycle is slow, prone to human error, and requires deep expertise, creating a significant bottleneck that delays release cycles.

## ✨ The Solution: Perf-AI Copilot

**Perf-AI Copilot** is an enterprise-grade multi-agent system designed to act as an intelligent assistant for performance engineers. It automates over 80% of the manual analysis effort by ingesting performance test data, detecting bottlenecks, generating AI-powered insights, and producing executive-ready reports in seconds.

## 🚀 Key Features

*   **Multi-Agent System:** A 5-agent pipeline orchestrates the entire analysis workflow.
*   **AI-Powered Root Cause Analysis:** Uses **Google's Gemini model** to synthesize findings and suggest actionable root causes.
*   **Intelligent Log Ingestion:** Automatically detects and parses both raw JMeter transaction logs (JTL) and pre-summarized Aggregate Reports.
*   **Automated Regression Analysis:** Compares current test results against the previous run to instantly identify performance improvements or regressions.
*   **Interactive Web UI:** A simple, clean user interface for uploading files and viewing reports.
*   **PDF Export:** Generates a professional, downloadable PDF version of the final report.
*   **Long-Term Memory:** A memory bank stores historical run data to enable trend analysis and regression testing.

## 🏗️ Architecture

The system is built on a sequential, 5-agent pipeline where each agent performs a specialized task and passes artifacts to the next agent in the chain.

```
┌──────────────────────┐      ┌──────────────────────┐      ┌──────────────────┐      ┌────────────────────┐      ┌───────────────────┐
│ 1. Ingestion Agent   ├─────►│ 2. Metric Agent      ├─────►│ 3. RC Agent      ├─────►│ 4. Report Agent    ├─────►│ 5. PDF Agent      │
│ (Parses CSV)         │      │ (Calculates Stats)   │      │ (Gemini-Powered) │      │ (Generates HTML)   │      │ (Generates PDF)   │
└──────────────────────┘      └──────────────────────┘      └──────────────────┘      └────────────────────┘      └───────────────────┘
```

1.  **Log Ingestion Agent:** Ingests the user-uploaded CSV file and uses a custom parser to intelligently detect the format (raw JTL vs. Aggregate Report).
2.  **Metric Analyzer Agent:** Calculates key performance indicators (p95, error rate, etc.), identifies the top 5 slowest transactions, and compares results against the previous run stored in the memory bank.
3.  **Root Cause (RC) Agent:** The "brain" of the system. It sends a detailed prompt—containing KPIs, slow transactions, and regression data—to the Gemini model for a deep root cause analysis.
4.  **Report Writer Agent:** Uses a Jinja2 template to render the AI summary, metric tables, and visualizations into a clean HTML report.
5.  **PDF Generator Agent:** Uses the `fpdf2` library to generate a portable, downloadable PDF version of the final report.

## 🛠️ Technology Stack

*   **Backend:** FastAPI, Uvicorn
*   **AI Integration:** `google-generativeai` (for Gemini)
*   **Data Handling:** Pandas, NumPy
*   **Reporting:** Jinja2, `markdown2`, `fpdf2`
*   **Frontend:** Vanilla HTML, CSS, and JavaScript

## ⚙️ Setup and Usage

### 1. Prerequisites
*   Python 3.9+
*   A Google Gemini API Key

### 2. Local Setup

1.  **Clone the repository:**
    ```sh
    git clone https://github.com/ajinsudhir/perf-ai-copilot.git
    cd perf-ai-copilot
    ```

2.  **Create and activate a virtual environment:**
    ```sh
    # Create the environment
    python -m venv venv

    # Activate it (on Windows PowerShell)
    .\venv\Scripts\Activate.ps1
    ```

3.  **Install dependencies:**
    ```sh
    pip install -r requirements.txt
    ```

4.  **Set up your environment variables:**
    -   Create a file named `.env` in the root directory.
    -   Add your Google Gemini API key to it:
        ```
        GOOGLE_API_KEY="YOUR_API_KEY_HERE"
        ```

### 3. Running the Application

1.  **Start the server:**
    ```sh
    uvicorn app.main:app --reload --port 8001
    ```

2.  **Access the UI:**
    -   Open your web browser and navigate to **`http://127.0.0.1:8001`**.
    -   Upload a sample JMeter CSV file (either a raw JTL or an Aggregate Report).
    -   Click "Analyze Performance" and watch the report generate in real-time.


### **Setup Step 1: Add Sample Data**

Since we don't have a UI, we need to provide a sample data file.

1.  Click on **"+ Add data"** in the right-hand sidebar of the notebook editor.
2.  Use the search bar to find and add the `sample_jmeter.csv` file you have on your local machine.
3.  It will be added to the `/kaggle/input/` directory. For this notebook, we'll assume it's named `aggregate_3010_500.csv`.


In [1]:
# Install all required libraries
!pip install -q pandas matplotlib google-generativeai markdown2 fpdf2


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.9/76.9 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.6/301.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 11.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-cloud-translate 3.12.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 5.29.5 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
pydrive2 1.21.3 requires cryptography<4

### **Setup Step 2: Configure Google API Key**

To protect your secret API key, we will use Kaggle Secrets.

1.  In the right-hand sidebar, click on the **"Add-ons"** menu, then select **"Secrets"**.
2.  Enter `GOOGLE_API_KEY` as the label.
3.  Paste your actual Google Gemini API key into the value field.
4.  Make sure the "Attached" toggle is on.


In [2]:
# General Imports
import os
import json
import uuid
import base64
import io
import logging
from typing import Dict, Any, Union
from threading import RLock

# Core Libraries
import pandas as pd
import matplotlib
matplotlib.use('Agg') # Use a non-interactive backend
import matplotlib.pyplot as plt
from jinja2 import Template
import markdown2
from fpdf import FPDF
import google.generativeai as genai

# Kaggle-specific import for secrets
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

# Set the API key from Kaggle Secrets
os.environ['GOOGLE_API_KEY'] = user_secrets.get_secret("GOOGLE_API_KEY")

print("Libraries imported and API key configured.")


Libraries imported and API key configured.


In [3]:
# General Imports
import os
import json
import uuid
import base64
import io
import logging
from typing import Dict, Any, Union
from threading import RLock
import tempfile

# Core Libraries
import pandas as pd
import matplotlib
matplotlib.use('Agg') # Use a non-interactive backend
import matplotlib.pyplot as plt
from jinja2 import Template
import markdown2
from fpdf import FPDF
import google.generativeai as genai

# Kaggle-specific import for secrets
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

# Set the API key from Kaggle Secrets
# Make sure you have a secret named "GOOGLE_API_KEY" in your notebook
os.environ['GOOGLE_API_KEY'] = user_secrets.get_secret("GOOGLE_API_KEY")

# ==============================================================================
# Storage Functions (formerly app/storage.py)
# ==============================================================================
SESSIONS: Dict[str, Dict[str, Any]] = {}
MEMORY_BANK: Dict[str, Dict[str, Any]] = {}
_lock = RLock()

def create_session(metadata: dict) -> str:
    sid = str(uuid.uuid4())
    with _lock:
        SESSIONS[sid] = {"id": sid, "created_at": "now", "metadata": metadata, "artifacts": {}, "status": "created"}
    return sid

def get_session(sid: str):
    with _lock:
        return SESSIONS.get(sid)

def update_session_status(sid: str, status: str):
    with _lock:
        session = get_session(sid)
        if session: session["status"] = status

def store_artifact(sid: str, name: str, data: Any):
    with _lock:
        session = get_session(sid)
        if session: session["artifacts"][name] = data

def push_to_memory(key: str, item: dict):
    with _lock:
        if key not in MEMORY_BANK: MEMORY_BANK[key] = {"history": []}
        MEMORY_BANK[key]["history"].append(item)

def get_last_run_from_memory(key: str) -> Union[Dict[str, Any], None]:
    with _lock:
        memory_entry = MEMORY_BANK.get(key)
        if memory_entry and memory_entry.get("history"): return memory_entry["history"][-1]
    return None

# ==============================================================================
# Base Agent (formerly app/agents/base_agent.py)
# ==============================================================================
logger = logging.getLogger("perf_ai")
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("[%(asctime)s] %(levelname)s %(name)s: %(message)s"))
    logger.addHandler(handler)
    logger.setLevel(logging.INFO)

class BaseAgent:
    def __init__(self, session_id: str, context: Dict[str, Any]):
        self.session_id = session_id
        self.context = context
    def run(self) -> Dict[str, Any]:
        raise NotImplementedError("Agent must implement run()")

# ==============================================================================
# JMeter Parser (formerly app/tools/jmeter_parser.py)
# ==============================================================================
def parse_jmeter_csv(path: str) -> Dict[str, Any]:
    logger.info(f"PARSER: Starting to read CSV file at {path}...")
    df = pd.read_csv(path)
    logger.info(f"PARSER: Successfully read {len(df)} rows.")
    df.columns = [c.strip() for c in df.columns]
    if '# Samples' in df.columns and '95% Line' in df.columns:
        logger.info("PARSER: Detected JMeter Aggregate Report format.")
        df.rename(columns={'Label': 'label', 'Average': 'elapsed_mean', '95% Line': 'elapsed_p95', 'Error %': 'error_rate_pct'}, inplace=True)
        df['error_rate'] = pd.to_numeric(df['error_rate_pct'].str.replace('%', '')) / 100
        return {"type": "summary", "df": df}
    else:
        logger.info("PARSER: Detected raw JMeter JTL format.")
        if "timeStamp" in df.columns:
            df["timeStamp"] = pd.to_datetime(df["timeStamp"], unit='ms')
        summary_df = df.describe(include='all')
        json_safe_summary = json.loads(summary_df.to_json(default_handler=str))
        return {"type": "raw", "df": df, "summary": json_safe_summary}

# ==============================================================================
# Log Ingestion Agent (formerly app/agents/ingestion_agent.py)
# ==============================================================================
class LogIngestionAgent(BaseAgent):
    def __init__(self, session_id: str, context: Dict[str, Any], file_path: str):
        super().__init__(session_id, context)
        self.file_path = file_path
    def run(self) -> Dict[str, Any]:
        logger.info(f"Ingesting file for session {self.session_id}: {self.file_path}")
        if not self.file_path.lower().endswith('.csv'):
            message = "Invalid file type. Please upload a CSV file."
            logger.error(f"Ingestion failed for session {self.session_id}: {message}")
            return {"status": "error", "message": message}
        try:
            parsed = parse_jmeter_csv(self.file_path)
            # FIX: Removed 'storage.' prefix
            store_artifact(self.session_id, "raw_dataset", parsed)
            update_session_status(self.session_id, "ingested")
            logger.info(f"INGESTION AGENT [{self.session_id}]: Ingestion successful.")
            if parsed.get("type") == "summary":
                return {"status": "ok", "rows": len(parsed["df"]), "summary": "Aggregate Report"}
            else:
                return {"status": "ok", "rows": len(parsed["df"]), "summary": parsed["summary"]}
        except Exception as e:
            logger.error(f"Failed to ingest file {self.file_path}: {e}")
            # FIX: Removed 'storage.' prefix
            update_session_status(self.session_id, "ingestion_failed")
            return {"status": "error", "message": str(e)}

# ==============================================================================
# Metric Analyzer Agent (formerly app/agents/metric_agent.py)
# ==============================================================================
def save_fig_to_base64(fig) -> str:
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight')
    plt.close(fig)
    return base64.b64encode(buf.getvalue()).decode('utf-8')

class MetricAnalyzerAgent(BaseAgent):
    def run(self) -> Dict[str, Any]:
        logger.info(f"Analyzing metrics for session {self.session_id}")
        # FIX: Removed 'storage.' prefix
        session = get_session(self.session_id)
        parsed = session["artifacts"].get("raw_dataset")
        if not parsed:
            return {"status": "error", "message": "No raw dataset found in session."}
        
        if parsed.get("type") == "summary":
            logger.info("Analyzing metrics from Aggregate Report summary.")
            df_summary = parsed["df"]
            results = {}
            total_row = df_summary[df_summary['label'].str.contains('TOTAL', case=False)]
            if not total_row.empty:
                total_metrics = total_row.iloc[0]
                results["elapsed_mean"] = float(total_metrics.get("elapsed_mean", 0))
                results["elapsed_p95"] = float(total_metrics.get("elapsed_p95", 0))
                results["error_rate"] = float(total_metrics.get("error_rate", 0))
                results["tps_mean"] = float(total_metrics.get("Throughput", 0))
            slowest_transactions = df_summary.sort_values(by='elapsed_p95', ascending=False).head(5)
            results["slowest_transactions"] = [{"label": row["label"], "p95_ms": int(row["elapsed_p95"])} for index, row in slowest_transactions.iterrows()]
            # FIX: Removed 'storage.' prefix
            store_artifact(self.session_id, "metrics", results)
        else:
            df = parsed["df"].copy()
            results = {}
            if "elapsed" in df.columns:
                fig = plt.figure(figsize=(10, 6))
                df["elapsed"].hist(bins=50)
                plt.title("Response Time Distribution (ms)")
                img_b64 = save_fig_to_base64(fig)
                # FIX: Removed 'storage.' prefix
                store_artifact(self.session_id, "plot_elapsed_hist_b64", img_b64)
                results["elapsed_mean"] = round(float(df["elapsed"].mean()), 2)
                results["elapsed_p95"] = round(float(df["elapsed"].quantile(0.95)), 2)
            if "elapsed" in df.columns and "label" in df.columns:
                slowest_transactions = df.groupby('label')['elapsed'].quantile(0.95).sort_values(ascending=False).head(5)
                results["slowest_transactions"] = [{"label": label, "p95_ms": round(p95_time)} for label, p95_time in slowest_transactions.items()]
            if "timeStamp" in df.columns:
                df.set_index("timeStamp", inplace=True)
                tps = df.resample("1s").size().rename("tps")
                results["tps_mean"] = round(float(tps.mean()), 2)
                fig = plt.figure(figsize=(10, 4))
                tps.plot()
                plt.title("Throughput (requests/sec)")
                img_tps_b64 = save_fig_to_base64(fig)
                # FIX: Removed 'storage.' prefix
                store_artifact(self.session_id, "plot_tps_b64", img_tps_b64)
            if "success" in df.columns:
                df["success_bool"] = df["success"].astype(str).str.lower().isin(["true", "1", "t"])
                results["error_rate"] = round(1.0 - df["success_bool"].mean(), 4)
            # FIX: Removed 'storage.' prefix
            store_artifact(self.session_id, "metrics", results)

        logger.info(f"Comparing with previous run for session {self.session_id}")
        # FIX: Removed 'storage.' prefix
        previous_run = get_last_run_from_memory("default")
        comparison_results = {}
        if previous_run and "metrics" in previous_run:
            prev_metrics = previous_run["metrics"]
            current_metrics = get_session(self.session_id)["artifacts"]["metrics"]
            comparison_results = {
                "previous_p95": prev_metrics.get("elapsed_p95"),
                "p95_delta": current_metrics.get("elapsed_p95", 0) - prev_metrics.get("elapsed_p95", 0),
                "previous_error_rate": prev_metrics.get("error_rate"),
                "error_rate_delta": current_metrics.get("error_rate", 0) - prev_metrics.get("error_rate", 0),
                "previous_tps_mean": prev_metrics.get("tps_mean"),
                "tps_mean_delta": current_metrics.get("tps_mean", 0) - prev_metrics.get("tps_mean", 0)
            }
        # FIX: Removed 'storage.' prefix
        store_artifact(self.session_id, "comparison", comparison_results)
        return {"status": "ok", "metrics": get_session(self.session_id)["artifacts"]["metrics"]}

# ==============================================================================
# Root Cause Agent (formerly app/agents/rc_agent.py)
# ==============================================================================
def call_llm(prompt: str) -> str:
    api_key = os.getenv("GOOGLE_API_KEY")
    if not api_key:
        return "LLM_DISABLED: Mocked root cause."
    try:
        genai.configure(api_key=api_key)
        # FIX: Corrected model name
        model = genai.GenerativeModel('gemini-1.5-flash-latest')
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"LLM_ERROR: {e}"

class RootCauseAgent(BaseAgent):
    def run(self) -> Dict[str, Any]:
        logger.info(f"Computing root cause for session {self.session_id}")
        # FIX: Removed 'storage.' prefix
        session = get_session(self.session_id)
        metrics = session["artifacts"].get("metrics", {})
        comparison = session["artifacts"].get("comparison", {})
        parsed = session["artifacts"].get("raw_dataset")
        df = parsed.get("df")
        findings = []
        if metrics.get("elapsed_p95", 0) > 2000:
            findings.append(f"High p95 latency of {metrics['elapsed_p95']:.0f}ms detected.")
        if metrics.get("error_rate", 0) > 0.05:
            findings.append(f"High error rate of {metrics['error_rate']:.2%} detected.")
        prompt = f"""
You are an expert Performance Engineer. Analyze the following test results.
**Key Automated Findings:**
- {"- ".join(findings) if findings else "No significant issues detected."}
**Detailed Metrics Summary:**
```json
{json.dumps(metrics, indent=2)}
```

**Analysis Request:**
1.  Synthesize the findings and metrics into a concise executive summary (2-3 sentences).
2.  Identify the top 2 most likely root causes for any performance issues.
3.  For each root cause, provide a brief, actionable recommendation for a developer or SRE to investigate.
4. Specifically analyze the 'Top 5 Slowest Transactions' and suggest possible reasons for their high response times (e.g., inefficient database query, external API call, complex business logic).
5. Based on the 'Comparison with Previous Run' data, comment on any significant performance regressions (e.g., increased p95 latency, higher error rate) or improvements. If a regression is detected, suggest it as a potential primary root cause.

Format your response in Markdown.
"""
        logger.info("Sending prompt to LLM for synthesis.")
        
        # Run the synchronous, blocking LLM call in a threadpool to avoid freezing the server
        llm_output = call_llm(prompt)

        result = {
            "heuristic_findings": findings,
            "llm_synthesis": llm_output
        }
       
        return {"status": "ok", "root_cause": result}


In [4]:
from IPython.display import display, HTML, FileLink

# --- Configuration ---
# This path has been updated to point to your uploaded data file.
FILE_PATH = '/kaggle/input/dataset/aggregate_3010_500.csv' 

# --- Main Execution Logic ---
def run_full_pipeline():
    # 1. Create a session
    session_id = create_session(metadata={"source": "Kaggle Notebook"})
    print(f"PIPELINE: Session created with ID: {session_id}\n")

    # 2. Run Ingestion Agent
    update_session_status(session_id, "ingesting")
    ingestion_agent = LogIngestionAgent(session_id, {}, FILE_PATH)
    ingestion_agent.run()
    print(f"PIPELINE: Ingestion complete. Status: {get_session(session_id)['status']}\n")

    # 3. Run Metric Analyzer Agent
    update_session_status(session_id, "analyzing_metrics")
    metric_agent = MetricAnalyzerAgent(session_id, {})
    metric_agent.run()
    print(f"PIPELINE: Metric analysis complete.\n")

    # 4. Run Root Cause Agent
    update_session_status(session_id, "analyzing_root_cause")
    rc_agent = RootCauseAgent(session_id, {})
    rc_agent.run()
    print(f"PIPELINE: Root cause analysis complete.\n")

    # 5. Run Report Writer Agent
    update_session_status(session_id, "generating_report")
    report_agent = ReportWriterAgent(session_id, {})
    report_agent.run()
    print(f"PIPELINE: HTML report generation complete.\n")
    
    # 6. Run PDF Generator Agent
    update_session_status(session_id, "generating_pdf")
    pdf_agent = PdfGeneratorAgent(session_id, {})
    pdf_agent.run()
    print(f"PIPELINE: PDF generation complete.\n")

    # 7. Finalize
    update_session_status(session_id, "completed")
    # Store the final metrics in the memory bank for the next run
    push_to_memory("default", get_session(session_id)['artifacts'])
    print(f"PIPELINE: Analysis successfully completed!")
    
    return session_id

# --- Execute and Display ---
try:
    final_session_id = run_full_pipeline()
    final_session = get_session(final_session_id)
    final_html_report = final_session['artifacts']['report_html']
    pdf_path = final_session['artifacts']['report_pdf_path']

    print("\n\n=================================================")
    print("           PERF-AI COPILOT REPORT              ")
    print("=================================================")
    
    # Provide a downloadable link for the PDF in the output directory
    print("\nPDF report has been saved to the output directory.")
    display(FileLink(pdf_path))
    
    print("\nDisplaying HTML report below:")

    # Display the final HTML report in the notebook output
    display(HTML(final_html_report))

except FileNotFoundError:
    print("\n\n--- ERROR ---")
    print(f"Could not find the data file at: {FILE_PATH}")
    print("Please make sure you have uploaded the data and that the FILE_PATH variable is correct.")
except Exception as e:
    print(f"\n\n--- An unexpected error occurred ---")
    print(e)



[2025-11-30 10:34:17,614] INFO perf_ai: Ingesting file for session 23ff9055-fa9b-4a24-87cd-682b00c0bd7c: /kaggle/input/dataset/aggregate_3010_500.csv
[2025-11-30 10:34:17,615] INFO perf_ai: PARSER: Starting to read CSV file at /kaggle/input/dataset/aggregate_3010_500.csv...
[2025-11-30 10:34:17,657] INFO perf_ai: PARSER: Successfully read 2010 rows.
[2025-11-30 10:34:17,665] INFO perf_ai: PARSER: Detected JMeter Aggregate Report format.
[2025-11-30 10:34:17,675] INFO perf_ai: INGESTION AGENT [23ff9055-fa9b-4a24-87cd-682b00c0bd7c]: Ingestion successful.
[2025-11-30 10:34:17,676] INFO perf_ai: Analyzing metrics for session 23ff9055-fa9b-4a24-87cd-682b00c0bd7c
[2025-11-30 10:34:17,677] INFO perf_ai: Analyzing metrics from Aggregate Report summary.
[2025-11-30 10:34:17,694] INFO perf_ai: Comparing with previous run for session 23ff9055-fa9b-4a24-87cd-682b00c0bd7c
[2025-11-30 10:34:17,695] INFO perf_ai: Computing root cause for session 23ff9055-fa9b-4a24-87cd-682b00c0bd7c
[2025-11-30 10:34:

PIPELINE: Session created with ID: 23ff9055-fa9b-4a24-87cd-682b00c0bd7c

PIPELINE: Ingestion complete. Status: ingested

PIPELINE: Metric analysis complete.

PIPELINE: Root cause analysis complete.



--- An unexpected error occurred ---
name 'ReportWriterAgent' is not defined
